# 2. Linear regression on house dataset
- Use this dataset on house price prediction in kaggle, 
- perform EDA on it and then predict the house prices using linear regression.
- You can drop the categorical features and the date columns. 
- Then try ElasticNetCV and RidgeCV on the same dataset. 
- Record the scores in a dataframe with the columns mae, mse, rmse so you can compare the models.

### EDA

In [565]:
import pandas as pd

df = pd.read_csv("../../data/data.csv")#) # make date column index and parse dates

df.head()

,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,sqft_above,sqft_basement,yr_built,yr_renovated,street,city,statezip,country
0,2014-05-02 00:00:00,313000.0,3.0,1.50,1340,7912,1.5,0,0,3,1340,0,1955,2005,18810 Densmore Ave N,Shoreline,WA 98133,USA
1,2014-05-02 00:00:00,2384000.0,5.0,2.50,3650,9050,2.0,0,4,5,3370,280,1921,0,709 W Blaine St,Seattle,WA 98119,USA
2,2014-05-02 00:00:00,342000.0,3.0,2.00,1930,11947,1.0,0,0,4,1930,0,1966,0,26206-26214 143rd Ave SE,Kent,WA 98042,USA
3,2014-05-02 00:00:00,420000.0,3.0,2.25,2000,8030,1.0,0,0,4,1000,1000,1963,0,857 170th Pl NE,Bellevue,WA 98008,USA
4,2014-05-02 00:00:00,550000.0,4.0,2.50,1940,10500,1.0,0,0,4,1140,800,1976,1992,9105 170th Ave NE,Redmond,WA 98052,USA


In [566]:
df.columns

Index(['date', 'price', 'bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot',
       'floors', 'waterfront', 'view', 'condition', 'sqft_above',
       'sqft_basement', 'yr_built', 'yr_renovated', 'street', 'city',
       'statezip', 'country'],
      dtype='object')

In [567]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4600 entries, 0 to 4599
Data columns (total 18 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   date           4600 non-null   object 
 1   price          4600 non-null   float64
 2   bedrooms       4600 non-null   float64
 3   bathrooms      4600 non-null   float64
 4   sqft_living    4600 non-null   int64  
 5   sqft_lot       4600 non-null   int64  
 6   floors         4600 non-null   float64
 7   waterfront     4600 non-null   int64  
 8   view           4600 non-null   int64  
 9   condition      4600 non-null   int64  
 10  sqft_above     4600 non-null   int64  
 11  sqft_basement  4600 non-null   int64  
 12  yr_built       4600 non-null   int64  
 13  yr_renovated   4600 non-null   int64  
 14  street         4600 non-null   object 
 15  city           4600 non-null   object 
 16  statezip       4600 non-null   object 
 17  country        4600 non-null   object 
dtypes: float

In [568]:
df = df.drop(["date", "street", "city", "statezip", "country"], axis="columns")
df.head()

,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,sqft_above,sqft_basement,yr_built,yr_renovated
0,313000.0,3.0,1.50,1340,7912,1.5,0,0,3,1340,0,1955,2005
1,2384000.0,5.0,2.50,3650,9050,2.0,0,4,5,3370,280,1921,0
2,342000.0,3.0,2.00,1930,11947,1.0,0,0,4,1930,0,1966,0
3,420000.0,3.0,2.25,2000,8030,1.0,0,0,4,1000,1000,1963,0
4,550000.0,4.0,2.50,1940,10500,1.0,0,0,4,1140,800,1976,1992


# inspecting values looking for 0 and extreme outliers

In [569]:
df["price"].sort_values().min()

np.float64(0.0)

In [570]:
# remove all 0 vals in price
df = df[df["price"] > 0]

In [571]:
# checking low vals
df["price"].sort_values(ascending=True).head()

4351     7800.0
1219    80000.0
1587    83000.0
4407    83300.0
4415    83300.0
Name: price, dtype: float64

In [572]:
# checking high
df["price"].sort_values(ascending=False).head()

4350    26590000.0
4346    12899000.0
2286     7062500.0
2654     4668000.0
2761     4489000.0
Name: price, dtype: float64

### options for further testing with extreme outliers filtered

In [573]:
# # calculating upper and lower limits by standard deviations and filtering out the outliers
# lower_limit = df["price"].mean() - 3 * df["price"].std()
# upper_limit = df["price"].mean() + 3 * df["price"].std()

# df = df[(df["price"] >= lower_limit) & (df["price"] <= upper_limit)]

# print(f"rows after filtering: {df.shape[0]} rows")

In [574]:
# # calculating upper and lower limits by percentiles and filtering out the outliers
# lower_limit = df["price"].quantile(0.01)
# upper_limit = df["price"].quantile(0.99)

# df = df[(df["price"] >= lower_limit) & (df["price"] <= upper_limit)]

# print(f"rows after filtering: {df.shape[0]} rows")
# print(f"lower : {lower_limit:,.0f}  |  upper: {upper_limit:,.0f}")


In [575]:
df.shape

(4551, 13)

In [576]:
df.describe()

,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,sqft_above,sqft_basement,yr_built,yr_renovated
count,4.551000e+03,4551.000000,4551.000000,4551.000000,4.551000e+03,4551.000000,4551.000000,4551.000000,4551.000000,4551.000000,4551.000000,4551.000000,4551.000000
mean,5.579059e+05,3.394639,2.155021,2132.372226,1.483528e+04,1.512195,0.006592,0.234674,3.449352,1822.221710,310.150516,1970.795649,808.564052
std,5.639299e+05,0.904595,0.776351,955.949708,3.596408e+04,0.538531,0.080932,0.765373,0.675160,854.452888,461.987629,29.760073,979.421487
min,7.800000e+03,0.000000,0.000000,370.000000,6.380000e+02,1.000000,0.000000,0.000000,1.000000,370.000000,0.000000,1900.000000,0.000000
25%,3.262643e+05,3.000000,1.750000,1460.000000,5.000000e+03,1.000000,0.000000,0.000000,3.000000,1190.000000,0.000000,1951.000000,0.000000
50%,4.650000e+05,3.000000,2.250000,1970.000000,7.680000e+03,1.500000,0.000000,0.000000,3.000000,1590.000000,0.000000,1976.000000,0.000000
75%,6.575000e+05,4.000000,2.500000,2610.000000,1.097800e+04,2.000000,0.000000,0.000000,4.000000,2300.000000,600.000000,1997.000000,1999.000000
max,2.659000e+07,9.000000,8.000000,13540.000000,1.074218e+06,3.500000,1.000000,4.000000,5.000000,9410.000000,4820.000000,2014.000000,2014.000000


### divide X, y

In [577]:
X, y = df.drop("price", axis = "columns"), df["price"]
X.head()

,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,sqft_above,sqft_basement,yr_built,yr_renovated
0,3.0,1.50,1340,7912,1.5,0,0,3,1340,0,1955,2005
1,5.0,2.50,3650,9050,2.0,0,4,5,3370,280,1921,0
2,3.0,2.00,1930,11947,1.0,0,0,4,1930,0,1966,0
3,3.0,2.25,2000,8030,1.0,0,0,4,1000,1000,1963,0
4,4.0,2.50,1940,10500,1.0,0,0,4,1140,800,1976,1992


In [578]:
y.head()

0     313000.0
1    2384000.0
2     342000.0
3     420000.0
4     550000.0
Name: price, dtype: float64

## Linear regression

In [579]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.33, random_state=42)

X_train.shape, y_train.shape

((3049, 12), (3049,))

### testing 2 types of scaling: MinMax and standard

In [580]:
# from sklearn.preprocessing import MinMaxScaler

# scaler = MinMaxScaler()
# scaler.fit(X_train)
# scaled_X_train = scaler.transform(X_train) 
# scaled_X_test = scaler.transform(X_test) 

# scaled_X_train.min(), scaled_X_train.max(), scaled_X_test.min(), scaled_X_test.max()

In [581]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaler.fit(X_train)
scaled_X_train = scaler.transform(X_train) 
scaled_X_test = scaler.transform(X_test) 

scaled_X_train.min(), scaled_X_train.max(), scaled_X_test.min(), scaled_X_test.max()

(np.float64(-3.7468293934934027),
 np.float64(29.481720895531357),
 np.float64(-3.611008266776248),
 np.float64(17.433910553624273))

In [582]:
from sklearn.linear_model import LinearRegression

model= LinearRegression()
model

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [583]:
model.fit(scaled_X_train, y_train)
model.coef_

array([-58914.01155502,  55822.55208177, 130685.38871054, -27834.78552674,
         9897.58147796,  37084.16235415,  39425.22087082,  31428.00187139,
       123109.51671601,  39961.0820985 , -70994.80298567,   6946.19161183])

In [584]:
model.intercept_

np.float64(563885.4278124447)

## Linear - prediction

In [585]:
test_sample_features = scaled_X_test[0].reshape(1, -1)
test_sample_target = y_test.values[0]
test_sample_features, test_sample_target


(array([[ 0.67511093,  3.06726711,  3.5094472 ,  2.42967554, -0.95746709,
         -0.07487897, -0.30352666, -0.66756108,  2.4463979 ,  2.67748162,
          1.03313568, -0.82111276]]),
 np.float64(1225000.0))

In [586]:
test_sample_features.shape

(1, 12)

In [587]:
test_sample_target

np.float64(1225000.0)

In [588]:
model.predict(test_sample_features)

array([1370257.4919046])

In [589]:
test_sample_target

np.float64(1225000.0)

prediction on test data

In [590]:
y_pred = model.predict(scaled_X_test)
y_pred.shape

(1502,)

In [591]:
y_test.shape

(1502,)

In [592]:
y_pred[:5]

array([1370257.4919046 ,  608251.99107972,  657977.77818934,
        279447.06287658,  522732.44380393])

In [593]:
y_test[:5].values

array([1225000.,  496752.,  612500.,  265000.,  615000.])

evaluation

In [594]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

def metrics(y_test, y_pred):
    MSE = mean_squared_error(y_test, y_pred)
    RMSE = np.sqrt(MSE)
    MAE = mean_absolute_error(y_test, y_pred)

    return {"MSE": MSE, "MAE": MAE, "RMSE": RMSE}

metrics(y_test, y_pred)

{'MSE': 53255714398.187004,
 'MAE': 158370.37304234505,
 'RMSE': np.float64(230771.99656411304)}

ridge_regression

In [595]:
from sklearn.linear_model import ElasticNetCV, RidgeCV

model_ridgeCV = RidgeCV(alphas=[.0001, .001, .01, .1, .5, 1, 5, 10], scoring="neg_mean_squared_error")
model_ridgeCV.fit(scaled_X_train, y_train)

model_ridgeCV.alpha_

np.float64(10.0)

In [596]:
y_pred = model_ridgeCV.predict(scaled_X_test)
metrics(y_test, y_pred)

{'MSE': 53212702836.396835,
 'MAE': 158279.42508212104,
 'RMSE': np.float64(230678.78714003338)}

In [597]:
model_ridgeCV.coef_

array([-58181.32863698,  55691.15972009, 130166.62441575, -27599.42948495,
        10030.17379053,  37060.09196888,  39525.16042177,  31410.6549112 ,
       122572.86205988,  39891.9721833 , -70515.62627721,   6989.42289946])

lasso regression

In [598]:
from sklearn.linear_model import LassoCV

model_lassoCV =LassoCV(alphas=100, cv=5, max_iter=10000)
model_lassoCV.fit(scaled_X_train, y_train)

model_lassoCV.alpha_

np.float64(1256.323941825215)

In [599]:
y_pred = model_lassoCV.predict(scaled_X_test)

metrics(y_test, y_pred)

{'MSE': 53109278402.24358,
 'MAE': 158044.74084528149,
 'RMSE': np.float64(230454.50397474028)}

In [600]:
model_lassoCV.coef_

array([-55312.04340294,  53229.99288274, 212376.81895368, -26233.96820334,
         8490.74538647,  36408.83664692,  39289.66244674,  29593.16973651,
        47117.4922694 ,      0.        , -69079.67462036,   5126.42203442])

elastic NET regression

In [601]:
model_elastic = ElasticNetCV(
    l1_ratio=[.1, .5, .7, .9, .95, .99, 1], alphas=100, max_iter=10000
)
model_elastic.fit(scaled_X_train, y_train)
model_elastic.l1_ratio_

np.float64(1.0)

In [602]:
model_elastic.alpha_

np.float64(1256.323941825215)

In [603]:
y_pred = model_elastic.predict(scaled_X_test)
metrics(y_test, y_pred)

{'MSE': 53109278402.24358,
 'MAE': 158044.74084528149,
 'RMSE': np.float64(230454.50397474028)}

In [604]:
model_elastic.coef_

array([-55312.04340294,  53229.99288274, 212376.81895368, -26233.96820334,
         8490.74538647,  36408.83664692,  39289.66244674,  29593.16973651,
        47117.4922694 ,      0.        , -69079.67462036,   5126.42203442])